# Exploracao de Dados - Passos Magicos (PEDE 2022-2024)

Analise exploratoria do dataset PEDE (Pesquisa Extensiva do Desenvolvimento Educacional) para o projeto de predicao de defasagem escolar.

**Objetivo:** Entender a distribuicao dos dados, identificar padroes, correlacoes e preparar insights para o modelo preditivo.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 60)

print(f"pandas: {pd.__version__}")

## 1. Carga dos Dados

O dataset principal tem 3 abas (PEDE2022, PEDE2023, PEDE2024), cada uma com nomes de colunas diferentes.

In [ ]:
DATA_PATH = "../data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

sheets = {}
for sheet in ["PEDE2022", "PEDE2023", "PEDE2024"]:
    sheets[sheet] = pd.read_excel(DATA_PATH, sheet_name=sheet)
    print(f"{sheet}: {sheets[sheet].shape[0]} linhas, {sheets[sheet].shape[1]} colunas")

In [ ]:
# Visao geral de cada aba
for name, df in sheets.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColunas: {list(df.columns)}")
    print(f"\nTipos:\n{df.dtypes.value_counts()}")
    print(f"\nMissing (top 10):\n{df.isnull().sum().sort_values(ascending=False).head(10)}")

## 2. Padronizacao e Combinacao

Usando o pipeline de preprocessing para padronizar os nomes e combinar os 3 anos.

In [ ]:
from src.preprocessing import load_data, combine_datasets, create_target, standardize_columns

raw_sheets = load_data(DATA_PATH)
df = combine_datasets(raw_sheets)
print(f"Dataset combinado: {df.shape[0]} linhas, {df.shape[1]} colunas")
print(f"\nColunas: {list(df.columns)}")
df.head()

In [ ]:
# Criar variavel-alvo binaria
df = create_target(df)
print(f"Distribuicao do target (defasagem binaria):")
print(df["target"].value_counts())
print(f"\nProporcao: {df['target'].mean():.1%} com defasagem")

## 3. Distribuicao da Variavel-Alvo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Target geral
df["target"].value_counts().plot.bar(ax=axes[0], color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Distribuicao do Target (geral)")
axes[0].set_xticklabels(["Sem Defasagem (0)", "Com Defasagem (1)"], rotation=0)

# Target por ano
df.groupby("ano")["target"].mean().plot.bar(ax=axes[1], color="#3498db")
axes[1].set_title("% Com Defasagem por Ano")
axes[1].set_ylabel("Proporcao")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 4. Analise dos Indicadores Educacionais

In [ ]:
indicadores = ["inde", "iaa", "ieg", "ips", "ida", "ipv", "ian"]
indicadores_presentes = [c for c in indicadores if c in df.columns]

print("Estatisticas dos indicadores:")
df[indicadores_presentes].describe().round(2)

In [ ]:
# Boxplots: indicadores por grupo de defasagem
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(indicadores_presentes):
    sns.boxplot(data=df, x="target", y=col, ax=axes[i], palette=["#2ecc71", "#e74c3c"])
    axes[i].set_title(col.upper())
    axes[i].set_xticklabels(["Sem Def.", "Com Def."])

# Esconder eixos extras
for j in range(len(indicadores_presentes), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Indicadores por Grupo de Defasagem", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Notas por Disciplina

In [ ]:
notas = ["nota_mat", "nota_por", "nota_ing"]
notas_presentes = [c for c in notas if c in df.columns]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate(notas_presentes):
    sns.histplot(data=df, x=col, hue="target", ax=axes[i], bins=20, palette=["#2ecc71", "#e74c3c"])
    axes[i].set_title(col.replace("nota_", "").upper())

plt.suptitle("Distribuicao de Notas por Grupo", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Correlacao entre Features

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Matriz de Correlacao")
plt.tight_layout()
plt.show()

In [ ]:
# Top correlacoes com o target
if "target" in corr_matrix.columns:
    target_corr = corr_matrix["target"].drop("target").sort_values(key=abs, ascending=False)
    print("Correlacao com o target (defasagem):")
    print(target_corr.head(15).to_string())

    plt.figure(figsize=(10, 6))
    target_corr.head(15).plot.barh(color=target_corr.head(15).map(lambda x: "#e74c3c" if x > 0 else "#2ecc71"))
    plt.title("Top 15 Features Mais Correlacionadas com Defasagem")
    plt.xlabel("Correlacao")
    plt.tight_layout()
    plt.show()

## 7. Distribuicao por PEDRA (Classificacao)

In [ ]:
if "pedra" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Contagem por pedra
    order = ["Quartzo", "Ágata", "Ametista", "Topázio"]
    pedras_presentes = [p for p in order if p in df["pedra"].values]
    sns.countplot(data=df, x="pedra", hue="target", order=pedras_presentes,
                  ax=axes[0], palette=["#2ecc71", "#e74c3c"])
    axes[0].set_title("Alunos por PEDRA e Defasagem")

    # % defasagem por pedra
    pedra_def = df.groupby("pedra")["target"].mean().reindex(pedras_presentes)
    pedra_def.plot.bar(ax=axes[1], color="#e74c3c")
    axes[1].set_title("% Defasagem por PEDRA")
    axes[1].set_ylabel("Proporcao")
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

    plt.tight_layout()
    plt.show()

## 8. Analise por Genero e Fase

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Defasagem por genero
if "genero" in df.columns:
    gen_def = df.groupby("genero")["target"].mean()
    gen_def.plot.bar(ax=axes[0], color=["#9b59b6", "#3498db"])
    axes[0].set_title("% Defasagem por Genero")
    axes[0].set_ylabel("Proporcao")
    axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

# Defasagem por fase
if "fase" in df.columns:
    fase_def = df.groupby("fase")["target"].mean().sort_index()
    fase_def.plot.bar(ax=axes[1], color="#e67e22")
    axes[1].set_title("% Defasagem por Fase")
    axes[1].set_ylabel("Proporcao")
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 9. Valores Faltantes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({"missing": missing, "pct": missing_pct})
missing_df = missing_df[missing_df["missing"] > 0].sort_values("missing", ascending=False)

print(f"Colunas com dados faltantes: {len(missing_df)} de {len(df.columns)}")
if len(missing_df) > 0:
    print(missing_df.to_string())

    plt.figure(figsize=(10, max(4, len(missing_df) * 0.3)))
    missing_df["pct"].plot.barh(color="#e74c3c")
    plt.title("% Dados Faltantes por Coluna")
    plt.xlabel("%")
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum dado faltante!")

## 10. Ponto de Virada e Indicacao para Bolsa

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, col in enumerate(["ponto_virada", "indicado_bolsa"]):
    if col in df.columns:
        ct = pd.crosstab(df[col], df["target"], normalize="index")
        ct.plot.bar(ax=axes[i], color=["#2ecc71", "#e74c3c"], stacked=True)
        axes[i].set_title(f"Defasagem vs {col}")
        axes[i].set_ylabel("Proporcao")
        axes[i].set_xticklabels(["Nao", "Sim"], rotation=0)
        axes[i].legend(["Sem Def.", "Com Def."])

plt.tight_layout()
plt.show()

## 11. Resumo / Conclusoes

**Insights principais:**

1. **Distribuicao do target** - verificar se classes sao balanceadas ou se sera necessario tratamento (oversampling, class_weight)
2. **Indicadores vs defasagem** - quais indicadores (INDE, IAA, IEG, etc.) tem mais poder discriminativo
3. **PEDRA** - classificacao ordinal fortemente correlacionada com defasagem (esperado, pois ambos derivam do INDE)
4. **Notas** - distribuicao entre alunos com/sem defasagem
5. **Dados faltantes** - IPP ausente em 2022, outras colunas com missing pontual
6. **Correlacoes** - features altamente correlacionadas que podem causar multicolinearidade

**Proximos passos:**
- Rodar o pipeline de treinamento: `docker compose --profile train run --rm train`
- Subir a API: `docker compose up app -d`